# 第 24 课：流式 CTC 系统综合实验与验收

这一课把前端、encoder contract、CTC 解码、PGS、RTF、语言模型和 WFST 思维连接起来。重点是建立可验证的系统接口。

<!-- course-upgrade-v2 -->
## 学习导航

| 项目 | 内容 |
|---|---|
| 所属阶段 | 系统综合 |
| 建议投入 | 3～5 小时，可分 2～3 次完成 |
| 前置要求 | 完成第 23 课；如果前测低于 2/3，先回看上一课小结 |
| 本课核心 | 系统 contract、端到端状态、验收矩阵 |
| 完成标准 | 能口头解释核心概念；独立完成强化题；从空白重写核心函数 |

高效顺序：**先回答前测 → 预测代码结果 → 再运行 → 修改一个变量 → 关闭答案复现 → 次日回忆。**


<!-- course-upgrade-v2 -->
## 课前诊断（先不要运行代码）

1. 分别用一句话解释：系统 contract、端到端状态、验收矩阵。
2. 画出这三个概念之间的输入—输出关系。
3. 写下你最不确定的一点，并给出一个暂时猜测。

自评：答对 0～1 题先复习前置课；答对 2 题可以正常学习；3 题都能讲清楚则直接挑战代码和迁移题。


<!-- course-bridge-v3 -->
## 知识接力：先取回旧知识，再进入本课

### 3 分钟闭卷回忆

在新 Markdown cell 中回答，**不要先翻前文**：CTC beam 中的候选前缀；概率与负对数代价方向；开发集和测试集权限。

- 三项都能用“含义 + 单位/shape + 一个数字例子”回答：进入本课。
- 能回答两项：学习本课，但把缺口记入 `LEARNING_LOG.md`。
- 只能回答零到一项：先回到 [上一课](23_流式Beam_WFST状态_Lattice与热词.ipynb)与[唯一学习路径](../LEARNING_PATH.md)，做一次最小实验；不要靠继续看新术语掩盖断点。

### 本课接口契约

```text
输入：声学候选、语言模型/图状态、分数权重
  ↓ 本课要学会的变换、状态或判断
输出：可解释、可冻结调参、可分块保持状态的上下文解码结果
```

学完后必须能解释：输入的哪个单位/shape/状态若丢失，会让输出“仍能运行却语义错误”。


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

def find_root():
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "pyproject.toml").exists(): return p
    raise FileNotFoundError("请从 learn_asr 或 notebooks 目录启动 Jupyter")

ROOT = find_root()
BLANK = "∅"
plt.rcParams["figure.figsize"] = (11, 4)
print("项目根目录:", ROOT)

import time
from collections import defaultdict
from ipywidgets import interact, IntSlider

## 1. 系统 contract

```text
accept_audio(samples)
  → 新特征帧
  → encoder logits + 新 cache
  → decoder active states
  → partial/stable/final event
```

每层都必须明确：输入时间单位、shape、有效长度、缓存所有权和 flush 行为。

## 2. 在真实音频时间轴上模拟声学后验

为了只检验流式系统逻辑，下面使用真实 0～9 拼接音频的时长，并构造可控 CTC 后验。它不是声学准确率实验；声学模型训练已在第 14 课单独完成。

In [ ]:
import soundfile as sf
y,sr=sf.read(ROOT/"data"/"spoken_digits_0_to_9_16k.wav")
duration=len(y)/sr;hop_ms=20;T=int(np.ceil(duration*1000/hop_ms));labels=[BLANK]+list("0123456789")
P=np.full((len(labels),T),.002);P[0,:]=.95
centers=np.linspace(10,T-10,10).astype(int)
for digit,c in zip("0123456789",centers):
    P[:,max(0,c-1):c+2]=.002;P[labels.index(digit),max(0,c-1):c+2]=.93
P/=P.sum(0,keepdims=True)
print(f"audio={duration:.2f}s frames={T} target=0123456789")

## 3. 流式 greedy + PGS 风格事件

In [ ]:
class GreedyStream:
    def __init__(self): self.prev=0;self.text=""
    def accept(self,chunk):
        old=self.text
        for x in chunk.argmax(0):
            if x!=0 and x!=self.prev:self.text+=labels[x]
            self.prev=x
        return {"pgs":"apd","text":self.text[len(old):],"full":self.text}

def run(chunk_frames):
    d=GreedyStream();events=[];t0=time.perf_counter()
    for s in range(0,T,chunk_frames): events.append(d.accept(P[:,s:s+chunk_frames]))
    elapsed=time.perf_counter()-t0
    return events,elapsed

events,elapsed=run(8)
for i,e in enumerate(events):
    if e["text"]: print(i,e)
print("RTF of decoder-only teaching loop:",elapsed/duration)

## 4. 交互比较 chunk 大小与更新频率

In [ ]:
@interact(chunk_frames=IntSlider(min=1,max=40,value=8,description="chunk frames"))
def inspect(chunk_frames=8):
    events,_=run(chunk_frames); changes=[(i+1,e["full"]) for i,e in enumerate(events) if e["text"]]
    print("chunk ms =",chunk_frames*hop_ms,"updates =",len(events),"text changes =",len(changes))
    print(changes)

## 5. 系统验收清单

### 正确性

- 离线与流式前端帧对齐；
- 不同 chunk 切法结果一致或差异有解释；
- repeated token、空音频、尾块、超长音频测试；
- PGS 重复、乱序和替换测试；
- CTC length audit。

### 性能

- RTF、first partial、first stable、final latency；
- P50/P90/P99；
- CPU/GPU/内存/并发；
- beam、LM scale、chunk、右上下文的准确率—延迟曲线。

### 质量

- CER/WER；
- 热词 recall 与 false trigger；
- 按噪声、说话人、语速、长度分桶分析。

## 6. 最终综合题

1. CTC、LM、WFST、PGS、RTF 各自解决什么问题？
2. 为什么一个 RTF=0.1 的系统仍可能有 1 秒首字延迟？
3. 为什么 CTC head 前面使用全局 Self-Attention 会破坏严格流式？
4. 如何验证 chunk cache 没有重复或遗漏？
5. 若热词召回提高但普通词误识别增加，应怎样评估？
6. 什么时候输出 partial，什么时候 stable，什么时候 final？

<details><summary>展开参考答案</summary>

1. CTC 处理未知对齐；LM 评价序列合理性；WFST 组织约束和加权搜索；PGS 表达增量修改；RTF 衡量计算相对音频时长。2. 可能等待大 chunk、右上下文、网络或稳定策略。3. 当前帧依赖尚未到达的未来。4. 与离线输出逐帧比较，并测试多种不规则 chunk。5. 同时报告热词 recall、false trigger、整体/分桶 CER-WER 和延迟。6. partial 可修订；stable 达到稳定条件；final 在 endpoint/结束且解码完成后提交。

</details>

## 学完后的下一阶段

你已经具备阅读和实现流式 CTC 解码器的概念地图。下一阶段不再增加零散名词，而是选择一个真实框架（如 WeNet/Kaldi 风格组件）做源码对应与完整工程实现。

<!-- course-upgrade-v2 -->
## 强化练习：第 24 课专属题库

请先把答案写进新的 Markdown/Code cell，再展开自评标准。

### A. 基础回忆

1. 不看上文，分别定义 `系统 contract`、`端到端状态`、`验收矩阵`。
2. 哪一个量/状态是本课最容易在模块边界丢失的？它的单位和 shape 是什么？
3. 本课至少写出两个“看起来能运行，但结果其实错误”的例子。

### B. 预测与推理

4. 场景：**单元模块都正确但时间戳整体漂移**。先预测现象，再说明原因，最后给出一项可以验证猜测的指标。
5. 改变本课最关键参数的 0.5×、1×、2×，分别预测准确率、延迟、内存或数值误差怎样变化。
6. 画一张最小数据流图，在每条边标出 dtype、shape、时间单位或概率/代价方向。

### C. 编程与排错

7. 编程任务：**为整条流式链路定义输入输出和回归测试**。至少加入正常、边界、错误输入三类测试。
8. 故意制造一个 off-by-one、shape、状态未 reset 或数值稳定性错误；记录错误现象和定位过程。
9. 不看本课实现，从空白 cell 重写最核心函数，并用原实现作数值对照。

### D. 迁移与表达

10. 跨课任务：**把前 23 课组织成一次可复现实验**。
11. 用 90 秒向没有学过 ASR 的人解释本课；禁止只念术语，必须举一个数字或生活例子。
12. 写出一个生产系统中会监控的指标，以及它异常时优先检查的三处位置。

<details><summary>展开自评标准</summary>

- 每题 0～2 分：0=无法回答；1=方向正确但缺少单位、边界或验证；2=解释完整且能用代码/数字验证。
- 24 分满分：达到 19 分再进入下一课；15～18 分次日重做错题；低于 15 分回看本课图和核心代码。
- 第 4 题必须包含“预测—原因—指标”，第 7～9 题必须真正运行测试，第 10 题必须明确上下游 contract。
- 核心答案至少应正确使用：系统 contract、端到端状态、验收矩阵。

</details>


<!-- course-upgrade-v2 -->
## 间隔复习与离场票

### 离场票（现在完成）

- [ ] 我能不用笔记解释 系统 contract、端到端状态、验收矩阵。
- [ ] 我能说出本课最常见的错误及其观测现象。
- [ ] 我能从空白重写一个核心函数，并通过至少 3 个测试。
- [ ] 我能说明本课对上一层和下一层接口的影响。

### 复习时间表

- **明天（5 分钟）**：闭卷写出三个核心概念和一个公式/shape。
- **7 天后（15 分钟）**：重做第 4、7、10 题，不运行原答案。
- **30 天后（20 分钟）**：从真实音频或随机张量重新构造一个最小实验。

把错题记录到根目录 `LEARNING_LOG.md`。不要只写“不会”，要写：原判断、证据、正确规则、下次检查动作。
